# Notebook 5 — Feature Engineering

**Rules:**
- Only use information available at prediction time (no columns derived from
  `order_delivered_customer_date` — that's the delivery outcome itself, leakage).
- Fit any transformer (encoder/scaler/imputer) on the **training split only**,
  then apply (`transform`, never `fit_transform`) to validation and test.
- Save every fitted object, not just the output table.

In [1]:
import pandas as pd
import numpy as np
import pickle
import os

ARTIFACTS_DIR = "artifacts"
train = pd.read_parquet(f"{ARTIFACTS_DIR}/03_train.parquet")
val   = pd.read_parquet(f"{ARTIFACTS_DIR}/03_val.parquet")
test  = pd.read_parquet(f"{ARTIFACTS_DIR}/03_test.parquet")

for c in ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date']:
    for df in (train, val, test):
        df[c] = pd.to_datetime(df[c])

print(train.shape, val.shape, test.shape)

(67533, 25) (14471, 25) (14472, 25)


## Build date-based features (available at prediction time)
`order_delivered_carrier_date` and `order_approved_at` happen before the
customer actually gets the package, so they are safe to use. Anything using
`order_delivered_customer_date` is NOT safe — that IS the outcome.

In [2]:
def add_date_features(df):
    df = df.copy()
    df['purchase_weekday'] = df['order_purchase_timestamp'].dt.dayofweek
    df['purchase_month'] = df['order_purchase_timestamp'].dt.month
    df['purchase_hour'] = df['order_purchase_timestamp'].dt.hour
    df['days_purchase_to_approved'] = (
        df['order_approved_at'] - df['order_purchase_timestamp']
    ).dt.total_seconds() / 86400
    df['days_purchase_to_carrier'] = (
        df['order_delivered_carrier_date'] - df['order_purchase_timestamp']
    ).dt.total_seconds() / 86400
    df['promised_delivery_days'] = (
        df['order_estimated_delivery_date'] - df['order_purchase_timestamp']
    ).dt.total_seconds() / 86400
    return df

train = add_date_features(train)
val = add_date_features(val)
test = add_date_features(test)
train[['purchase_weekday','purchase_month','purchase_hour',
       'days_purchase_to_approved','days_purchase_to_carrier',
       'promised_delivery_days']].head()

,purchase_weekday,purchase_month,purchase_hour,days_purchase_to_approved,days_purchase_to_carrier,promised_delivery_days
0,3,9,12,0.000000,53.205035,18.488449
1,0,10,9,3.254213,20.178738,23.593866
2,0,10,16,2.963125,17.983981,34.293866
3,0,10,21,0.553657,21.633877,52.123831
4,0,10,21,1.248762,21.614155,56.115556


## Define feature groups

In [3]:
NUMERIC_FEATURES = [
    'n_items', 'n_unique_products', 'n_unique_sellers', 'total_price',
    'total_freight', 'avg_item_price', 'n_payment_installments',
    'total_payment_value', 'n_payment_rows',
    'purchase_weekday', 'purchase_month', 'purchase_hour',
    'days_purchase_to_approved', 'days_purchase_to_carrier',
    'promised_delivery_days',
]
CATEGORICAL_FEATURES = ['main_payment_type', 'main_category', 'customer_state']
NUMERIC_FEATURES = [c for c in NUMERIC_FEATURES if c in train.columns]
CATEGORICAL_FEATURES = [c for c in CATEGORICAL_FEATURES if c in train.columns]
LABEL = 'is_late'

print("numeric:", NUMERIC_FEATURES)
print("categorical:", CATEGORICAL_FEATURES)

numeric: ['n_items', 'n_unique_products', 'n_unique_sellers', 'total_price', 'total_freight', 'avg_item_price', 'n_payment_installments', 'total_payment_value', 'n_payment_rows', 'purchase_weekday', 'purchase_month', 'purchase_hour', 'days_purchase_to_approved', 'days_purchase_to_carrier', 'promised_delivery_days']
categorical: ['main_payment_type', 'main_category', 'customer_state']


## Handle missing values (fit imputer on TRAIN only)

In [4]:
from sklearn.impute import SimpleImputer

num_imputer = SimpleImputer(strategy='median')
train[NUMERIC_FEATURES] = num_imputer.fit_transform(train[NUMERIC_FEATURES])
val[NUMERIC_FEATURES]   = num_imputer.transform(val[NUMERIC_FEATURES])
test[NUMERIC_FEATURES]  = num_imputer.transform(test[NUMERIC_FEATURES])

for c in CATEGORICAL_FEATURES:
    train[c] = train[c].fillna('missing')
    val[c] = val[c].fillna('missing')
    test[c] = test[c].fillna('missing')

## Encode categoricals (fit on TRAIN only)

In [5]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe.fit(train[CATEGORICAL_FEATURES])

def encode(df):
    arr = ohe.transform(df[CATEGORICAL_FEATURES])
    cols = ohe.get_feature_names_out(CATEGORICAL_FEATURES)
    return pd.DataFrame(arr, columns=cols, index=df.index)

train_cat = encode(train)
val_cat = encode(val)
test_cat = encode(test)
train_cat.head()

,main_payment_type_boleto,main_payment_type_credit_card,main_payment_type_debit_card,main_payment_type_missing,main_payment_type_voucher,main_category_agro_industry_and_commerce,main_category_air_conditioning,main_category_art,main_category_arts_and_craftmanship,main_category_audio,...,customer_state_PR,customer_state_RJ,customer_state_RN,customer_state_RO,customer_state_RR,customer_state_RS,customer_state_SC,customer_state_SE,customer_state_SP,customer_state_TO
0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


## Scale numeric features (fit on TRAIN only)

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
train_num = pd.DataFrame(scaler.fit_transform(train[NUMERIC_FEATURES]),
                          columns=NUMERIC_FEATURES, index=train.index)
val_num = pd.DataFrame(scaler.transform(val[NUMERIC_FEATURES]),
                        columns=NUMERIC_FEATURES, index=val.index)
test_num = pd.DataFrame(scaler.transform(test[NUMERIC_FEATURES]),
                         columns=NUMERIC_FEATURES, index=test.index)

## Assemble final feature tables

In [7]:
X_train = pd.concat([train_num, train_cat], axis=1)
X_val   = pd.concat([val_num, val_cat], axis=1)
X_test  = pd.concat([test_num, test_cat], axis=1)

y_train = train[LABEL].values
y_val   = val[LABEL].values
y_test  = test[LABEL].values

print(X_train.shape, X_val.shape, X_test.shape)
FEATURE_LIST = list(X_train.columns)
print("n_features:", len(FEATURE_LIST))

(67533, 119) (14471, 119) (14472, 119)
n_features: 119


## Artifact: the final feature table, the fitted transformers, and the feature list
In production the pipeline must LOAD these same fitted objects and never
re-fit them on new data.

In [8]:
X_train.assign(is_late=y_train).to_parquet(f"{ARTIFACTS_DIR}/05_train_features.parquet", index=False)
X_val.assign(is_late=y_val).to_parquet(f"{ARTIFACTS_DIR}/05_val_features.parquet", index=False)
X_test.assign(is_late=y_test).to_parquet(f"{ARTIFACTS_DIR}/05_test_features.parquet", index=False)

with open(f"{ARTIFACTS_DIR}/05_imputer.pkl", "wb") as f:
    pickle.dump(num_imputer, f)
with open(f"{ARTIFACTS_DIR}/05_encoder.pkl", "wb") as f:
    pickle.dump(ohe, f)
with open(f"{ARTIFACTS_DIR}/05_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)
with open(f"{ARTIFACTS_DIR}/05_feature_list.pkl", "wb") as f:
    pickle.dump(FEATURE_LIST, f)

print("Saved feature tables + imputer + encoder + scaler + feature_list to", ARTIFACTS_DIR)

Saved feature tables + imputer + encoder + scaler + feature_list to artifacts
